# Kaggle GPU Notebook for NAFLD Image Training

这个 notebook 直接复用仓库里的 `src/liver_ml/image_train.py`，适合把整个项目作为 Kaggle Dataset 挂载后在 GPU 上训练。

默认对比 **simple_cnn / ResNet50 / EfficientNet-B0 / DenseNet121 / ViT-B/16**，以及本地 **`yolo_model/yolo11s-seg.pt`、`yolo_model/yolo8s-seg.pt`** 对应的 **`yolo11s_seg`、`yolo8s_seg`**（见 `src/liver_ml/yolo_model.py`）。

In [ ]:
from pathlib import Path
import os
import shutil

REPO_INPUT = Path('/kaggle/input/liver-test')
WORKDIR = Path('/kaggle/working/liver_test')

if not WORKDIR.exists():
    shutil.copytree(REPO_INPUT, WORKDIR)

os.chdir(WORKDIR)
print('workdir =', WORKDIR)
print('gpu available =', __import__('torch').cuda.is_available())


In [ ]:
import importlib.util
import subprocess
import sys

# Kaggle 等环境若未预装 ultralytics（YOLO 分类），自动安装
if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

sys.path.insert(0, str(WORKDIR / "src"))

from liver_ml.image_train import ImageTrainConfig, train_image_models, predict_images

# 与本地仓库一致：输出写到项目根下 `outputs/`（便于与本地训练结果合并）
OUTPUT_ROOT = WORKDIR / "outputs"

config = ImageTrainConfig(
    data_dir=WORKDIR / "data" / "image" / "B-MODE",
    output_root=OUTPUT_ROOT,
    image_size=224,
    batch_size=16,
    epochs=12,
    lr=1e-4,
    num_workers=2,
    min_epoch_samples=1024,
    models=(
        "simple_cnn",
        "resnet50",
        "efficientnet_b0",
        "densenet121",
        "vit_b_16",
        "yolo11s_seg",
        "yolo8s_seg",
    ),
)
config


In [ ]:
# 首次建议先 smoke test
# config.models = ('simple_cnn',)
# config.epochs = 1

leaderboard = train_image_models(config)
leaderboard


In [ ]:
best_model = leaderboard.iloc[0]["model"]
checkpoint = WORKDIR / "outputs" / "models" / "image" / f"{best_model}.pt"
sample_images = sorted((WORKDIR / "data" / "image" / "B-MODE" / "NAFLD").glob("*.png"))[:2]
predict_images(checkpoint, sample_images)


## Kaggle 使用说明

1. 把整个项目目录上传为 Kaggle Dataset，假设名字叫 `liver-test`。
2. Notebook 右侧打开 GPU。
3. 如果 Dataset 名字不是 `liver-test`，修改 `REPO_INPUT`。
4. 如果只上传了图像数据而没有上传整个仓库，就把这个 notebook 里的 `REPO_INPUT` 和 `config.data_dir` 改成你的实际挂载路径。
5. 训练输出与本地目录结构一致：写到 **`/kaggle/working/liver_test/outputs/`**（即 `outputs/models/image/`、`outputs/reports/image/`、`outputs/figures/image/`），不再使用单独的 `nafld_outputs` 目录，便于与本地 `outputs/` 合并对比。
